# CDD-11-30: A3-L duration diagnostic

A3-L repeats A3 from scratch with the same SIDD32 model, seed, losses, crop size, and two-T4 DDP setup, but extends the cosine schedule from 5 to 20 epochs. It logs precision, recall, F1, AUROC, and average precision for every degradation label and evaluates two checkpoints: best restoration PSNR and best degradation macro-F1. Expected Kaggle runtime is approximately 25–35 minutes for training plus a few minutes for the two full-frame evaluations.

Only the existing `cdd-11-30` and `nafnetmodel` Kaggle inputs are required. The held-out test split is not evaluated.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("CWD:", Path.cwd())

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_a3l")
CONFIG = Path("configs/calibration_a3_long.json")
RUN_NAME = "a3l_degradation_supervision_sidd32_seed42_20ep"
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
gpu_names = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
] , text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), (
    f"Select the Kaggle 2xT4 accelerator; found: {gpu_names}"
)
print("GPUs:", gpu_names)
print("CDD-11:", CDD11_ROOT)
print("Pretrained files:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
# Recheck pairs, scene isolation, and exact pretrained compatibility.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit_a3l.json",
], check=True)

In [ ]:
# Change this to True only after the audit cell succeeds.
RUN_A3L = False

In [ ]:
runner_command = [
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
    "--runs", RUN_NAME,
]
if RUN_A3L:
    subprocess.run(runner_command, check=True)
else:
    subprocess.run([*runner_command, "--dry-run"], check=True)
    print("Dry run complete. Set RUN_A3L = True and rerun from the safety-switch cell.")

In [ ]:
from IPython.display import display
import pandas as pd

run_dir = EXPERIMENTS_ROOT / RUN_NAME
for label, path in {
    "training": run_dir / "run_summary.json",
    "best restoration": run_dir / "evaluation" / "summary.json",
    "best reasoning": run_dir / "reasoning_evaluation" / "summary.json",
}.items():
    if path.is_file():
        print(f"\n{label}:")
        display(json.loads(path.read_text()))
train_log = run_dir / "train_log.csv"
if train_log.is_file():
    display(pd.read_csv(train_log))

In [ ]:
# Create the two files needed for the next analysis.
import zipfile
from IPython.display import FileLink, FileLinks

summary_csv = EXPERIMENTS_ROOT / "ablation_summary.csv"
if summary_csv.is_file():
    display(pd.read_csv(summary_csv))
    lightweight = Path("/kaggle/working/a3l_lightweight_results.zip")
    with zipfile.ZipFile(lightweight, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for path in sorted(EXPERIMENTS_ROOT.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {".pt", ".png", ".jpg", ".jpeg"}:
                output_zip.write(path, path.relative_to(EXPERIMENTS_ROOT))

    comparisons = Path("/kaggle/working/a3l_comparisons.zip")
    with zipfile.ZipFile(comparisons, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for evaluation_name in ("evaluation", "reasoning_evaluation"):
            comparison_root = run_dir / evaluation_name / "comparisons"
            for path in sorted(comparison_root.glob("*.png")):
                output_zip.write(path, Path(evaluation_name) / path.name)

    print("Download both ZIP files and send them back for A3-L analysis:")
    display(FileLink(str(lightweight)))
    display(FileLink(str(comparisons)))
else:
    print("No completed A3-L summary yet.")

if EXPERIMENTS_ROOT.is_dir():
    display(FileLinks(str(EXPERIMENTS_ROOT)))